# Phase 3: Explainability Audit & Prescriptive Decision Engine
### Customer Churn Decision Engine
**Objective:** Audit global and local feature importance via SHAP TreeExplainer, render individual root-cause waterfall plots, and map SHAP attributions into automated retention playbooks.

In [ ]:
import sys
sys.path.append("..")
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from src.config import PIPELINE_ARTIFACT_PATH, METADATA_ARTIFACT_PATH, RAW_DATA_DIR
from src.models.pipeline import load_pipeline_artifacts
from src.data.preprocessor import load_raw_dataset, split_data
from src.explainability.shap_service import (
    explain_single_customer,
    render_customer_waterfall_figure,
    humanize_feature_name,
)
from src.decision_engine.rules import prescribe_retention_action

# 1. Load pipeline and metadata
pipeline, metadata = load_pipeline_artifacts(PIPELINE_ARTIFACT_PATH, METADATA_ARTIFACT_PATH)
feature_names = metadata["encoded_feature_names"]
optimal_threshold = metadata["optimal_threshold"]
print(f"Loaded {metadata['model_name']} with calibrated threshold τ* = {optimal_threshold}")

## 1. Local Customer Diagnostic & Waterfall Explanation

In [ ]:
X, y = load_raw_dataset()
sample_customer = X.iloc[[2]]  # High risk customer
cust_id = sample_customer['customerID'].values[0] if 'customerID' in sample_customer else 'CUST-001'

# Compute TreeExplainer
from src.explainability.shap_service import load_shap_explainer
explainer = load_shap_explainer()

explanation = explain_single_customer(
    pipeline=pipeline,
    explainer=explainer,
    customer_raw_df=sample_customer,
    feature_names=feature_names,
    top_k=5
)

print(f"=== Customer {cust_id} Diagnostics ===")
print(f"Predicted Churn Risk: {explanation['prediction_probability'] * 100:.1f}%")
print("\nTop Positive Risk Drivers:")
for driver in explanation["risk_drivers"]:
    print(f"  • {driver['display_name']} (SHAP: +{driver['shap_value']:.4f})")

## 2. Prescriptive Action Recommendation

In [ ]:
action = prescribe_retention_action(
    churn_prob=explanation["prediction_probability"],
    top_shap_drivers=explanation["risk_drivers"]
)

print(f"Action Code:        {action['action_code']}")
print(f"Playbook Headline:  {action['action_title']}")
print(f"Intervention Level: {action['priority']}")
print(f"Estimated Cost:     {action['estimated_cost']}")
print(f"Trigger Rationale:  {action['trigger_rationale']}")